# Backend Verification for the wafer.space GF180MCU Run-1

[D. Mitch Bailey](https://www.linkedin.com/in/mitch-bailey-cvc/), [ShuhariSystem](https://www.shuharisystem.com/)

H. Ojiro, [ShuhariSystem](https://www.shuharisystem.com/)

# Overview
This notebook performs backend LVS (with [klayout](https://github.com/KLayout/klayout) or [magic](https://github.com/RTimothyEdwards/magic)/[netgen](https://github.com/RTimothyEdwards/netgen)), soft-connection checks (with magic/netgen) and/or reliability verfication (with [CVC-RV](https://github.com/d-m-bailey/cvc)) with open source EDA software and systems.

# wafer.space GF180MCU Run 1
The 38 submissions to wafer.space shuttle for ws-run1 listed below are located [here](https://github.com/wafer-space/ws-run1).

| No | Code | Chip Top |
| -- | ---- | -------- |
| 1 | 2975 | 2975_chip_top_0_6 |
| 2 | 2975 | 2975_chip_top_2_6 |
| 3 | AS03 | AS03_chip_top_6_0 |
| 4 | BRWN | BRWN_ENGN2912E_TOP_8_2 |
| 5 | BTAP | BTAP_chip_top_0_0 |
| 6 | BTAP | BTAP_chip_top_2_0 |
| 7 | CAFE | CAFE_chip_top_12_0 |
| 8 | CHES | CHES_chip_top_10_4 |
| 9 | GD02 | GD02_chip_top_14_4 |
| 10| GD03 | GD03_chip_top_6_6 |
| 11 | GD04 | GD04_chip_top_10_8 |
| 12 | GD04 | GD04_chip_top_6_8 |
| 13 | HZ80 | HZ80_chip_top_14_0 |
| 14 | ISHI | ISHI_ISHI-KAI_WS_RUN1_12_4 |
| 15 | JKU1 | JKU1_chip_top_10_0 |
| 16 | JKU2 | JKU2_chip_top_0_8 |
| 17 | KIAN | KIAN_chip_top_8_0 |
| 18 | MOLE | MOLE_chip_top_8_4 |
| 19 | MOS2 | MOS2_chip_top_10_6 |
| 20 | MOSB | MOSB_chip_top_4_0 |
| 21 | MOSB | MOSB_chip_top_4_2 |
| 22 | OCD1 | OCD1_caravel_openframe_top_8_6 |
| 23 | OCD2 | OCD2_gf180mcu_ocd_sram_top_2_8 |
| 24 | OCD2 | OCD2_gf180mcu_ocd_sram_top_4_8 |
| 25 | RBOY | RBOY_chip_top_12_6 |
| 26 | RZ80 | RZ80_chip_top_0_2 |
| 27 | RZ80 | RZ80_chip_top_2_2 |
| 28 | RZML | RZML_chip_top_12_2 |
| 29 | TQVA | TQVA_chip_top_14_8 |
| 30 | TQVB | TQVB_chip_top_14_6 |
| 31 | TQVC | TQVC_chip_top_12_8 |
| 32 | TQVC | TQVC_chip_top_8_8 |
| 33 | TRID | TRID_TOP_14_2 |
| 34 | TTP2 | TTP2_tt_gf_wrapper_6_4 |
| 35 | TTPG | TTPG_tt_gf_wrapper_6_2 |
| 36 | TZ01 | TZ01_chip_top_0_4 |
| 37 | TZ01 | TZ01_chip_top_2_4 |
| 38 | WSLG | WSLG_chip_top_10_2 |

## I. Set initial environment variables.

In [ ]:
%%writefile /content/env
export LOCAL_INSTALL=/content/local
export PATH=$PATH:$LOCAL_INSTALL/bin
export PDK_ROOT=/content/pdks
export PDK=gf180mcuD
export PDKPATH=$PDK_ROOT/$PDK
export PDK_COMMIT=f6eeac7dad085ffcc829ccfd721f7b4ce39edcf7
export MAGIC_VERSION=8.3.688
export NETGEN_VERSION=1.5.323
export KLAYOUT_DISTRO=Ubuntu-24
export KLAYOUT_DEB=klayout_0.30.12-1_amd64.deb
export EXTRA_CHECK_VERSION=wafer.space-run1
export CVC_VERSION=master
export DATA_ROOT=/content/design_data
export CHECK_VERSION=main
export LVS_ROOT=/root/extra_be_checks
export MPW_ROOT=/content/ws-mpw-be
export MPW=ws-run1
export SHUTTLE_ID=G801
if [[ -f $DATA_ROOT/project_env ]]; then
  cat $DATA_ROOT/project_env
  source $DATA_ROOT/project_env
fi

# II. Program installation.
Only needs to be executed once.

This step sets up the pdk and installs magic, klayout, netgen and cvc_rv.

Runtime: 3-4 minutes

Note: klayout LVS rule files are modified as follows:


In [ ]:
%%shell
cd
cat /content/env
source /content/env

lsb_release -a 2>/dev/null

sudo apt update

if ! command -v ciel; then
  echo "==> Installing ciel..."
  pip install ciel
fi
rm -rf $PDK_ROOT
ciel enable --pdk $PDK $PDK_COMMIT
echo "==> Using pdk $PDK commit $PDK_COMMIT (patched)"

echo "==> Downloading extra_be_checks $EXTRA_CHECK_VERSION"
rm -rf extra_be_checks
git clone https://github.com/d-m-bailey/extra_be_checks.git -b $EXTRA_CHECK_VERSION
echo "==> Using extra_be_checks commit $(cd extra_be_checks; git rev-parse HEAD) "

: <<COMMENTS
echo "==> Modify run_extract"
echo "    - gensub function is not supported by awk"
echo "    - Remove lef write option -pinonly"
echo "    - Append check_extraction command"
sed -i.bak \
  -e '/ gensub/ s/print gensub(\/\.\*, layer=\/, "layer=", 1)/sub(\/\.\*, layer=\/, "layer="); print/' \
  -e '/lef write / s/ -.*//' \
  -e '$a\
echo "Running check_extraction .."\
${LVS_ROOT}/check_extraction ${EXT_DIR}
' \
  "${LVS_ROOT}/run_extract"
COMMENTS

echo "==> Modify run_extract"
echo "    - gensub function is not supported by awk"
echo "    - Using gf180mcu_fd_ip_sram{64|128|256|512}.lef files in libs.ref"
echo "    - Using gf180mcu_fd_io__*.lef files in libs.ref"
echo "    - Append check_extraction command"
sed -i.bak \
-e '/ gensub/ s/print gensub(\/\.\*, layer=\/, "layer=", 1)/sub(\/\.\*, layer=\/, "layer="); print/' \
-e '/^abstract_status/a\
\
shopt -s nullglob\
\
for sram_lef in "${EXT_DIR}"/gf180mcu_fd_ip_sram*.lef; do\
  mv -v "${sram_lef}" "${sram_lef}".bak\
  base=$(basename "${sram_lef}")\
  base_name=${base%.lef}\
  macro_name=${base%\\$*}\
  echo "Copying ${PDKPATH}/libs.ref/gf180mcu_fd_ip_sram/lef/${macro_name}.lef -> ${sram_lef} .."\
  echo "Modify macro name ${macro_name} -> ${base_name}"\
  sed -e "s/${macro_name}/${base_name}/g" \
         "${PDKPATH}/libs.ref/gf180mcu_fd_ip_sram/lef/${macro_name}.lef"\
         > "${sram_lef}"\
done\
\
for io_lef in "${EXT_DIR}"/gf180mcu_fd_ip_io__*.lef; do\
  mv -v "${io_lef}" "${io_lef}".bak\
  base=$(basename "${io_lef}")\
  base_name=${base%.lef}\
  macro_name=${base%\\$*}\
  echo "Copying ${PDKPATH}/libs.ref/gf180mcu_fd_ip_io/lef/${macro_name}.lef -> ${io_lef} .."\
  echo "Modify macro name ${macro_name} -> ${base_name}"\
  sed -e "s/${macro_name}/${base_name}/g" \
         "${PDKPATH}/libs.ref/gf180mcu_fd_ip_io/lef/${macro_name}.lef"\
         > "${io_lef}"\
done\
\
shopt -u nullglob
' \
-e '$a\
echo "Runinng check_extraction .."\
${LVS_ROOT}/check_extraction ${EXT_DIR}
' ${LVS_ROOT}/run_extract

if ! command -v netgen; then
  echo "==> Downloading and installing netgen $NETGEN_VERSION"
  git clone https://github.com/RTimothyEdwards/netgen.git --depth=1 -b $NETGEN_VERSION
  cd netgen
  ./configure --prefix=$LOCAL_INSTALL
  make
  make install
  cd
fi
echo "==> Using netgen version $(netgen -batch | awk '/Netgen/ {print $2}')"

if ! command -v magic; then
  echo "==> Downloading and installing magic $MAGIC_VERSION"
  git clone https://github.com/RTimothyEdwards/magic.git --depth=1 -b $MAGIC_VERSION
  cd magic
  ./configure --prefix=$LOCAL_INSTALL
  make
  make install
  cd
fi
echo "==> Using magic version $(magic -dnull -noc --version)"

if ! command -v klayout; then
  echo "==> Downloading and installing klayout $KLAYOUT_DEB for $KLAYOUT_DISTRO"
  wget -P /tmp https://www.klayout.org/downloads/$KLAYOUT_DISTRO/$KLAYOUT_DEB
  sudo dpkg -i /tmp/$KLAYOUT_DEB
  sudo apt --fix-broken install
  pip install docopt
  pip install klayout
  cd
fi
echo "==> Using $(klayout -v)"

if ! command -v cvc_rv; then
  echo "==> Downloading and installing cvc_rv $CVC_VERSION"
  sudo apt install autopoint bison flex
  git clone https://github.com/d-m-bailey/cvc --depth=1 -b $CVC_VERSION
  cd cvc
  autoreconf -vif
  ./configure --prefix=$LOCAL_INSTALL --disable-nls
  make
  make install
  cd
fi
echo "==> Using $(cvc_rv -v)"

echo "#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~"
echo "pdk $PDK commit $PDK_COMMIT (patched)"
echo "extra_be_checks commit $(cd extra_be_checks; git rev-parse HEAD)"
echo "netgen version $(netgen -batch | awk '/Netgen/ {print $2}')"
echo "magic version $(magic -dnull -noc --version)"
echo "$(klayout -v)"
echo "$(cvc_rv -v)"
echo "#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~"

# III. Create the design repo and extract the GDS file for each design.

Run to refresh the design data.

Runtime: ~10 minutes.

In [ ]:
%%shell
cat /content/env
source /content/env
: <<COMMENTS
COMMENTS
# rm -rfv ${DATA_ROOT} ${MPW_ROOT}
if [[ ! -d ${DATA_ROOT} ]]; then
  git clone https://github.com/H-Ojiro/ws-mpw-be.git --depth=1 -b ${CHECK_VERSION} ${DATA_ROOT}
fi
if [[ ! -d ${MPW_ROOT} ]]; then
  git clone https://github.com/wafer-space/$MPW.git --depth=1 ${DATA_ROOT}
fi

# Assemble G801.oas from the individual files
pushd ${DATA_ROOT}/layout
if [[ ! -e ${SHUTTLE_ID}.oas ]]; then
  echo "Merging ${SHUTTLE_ID}.oas .."
  ./create-reticle-oas.sh
fi
ls -l *.oas

# Extract gds file of each mpw chip
pushd ${MPW_ROOT}
cd layout
echo "Creating a chip gds file .."
${MPW_ROOT}/scripts/create_chip_gds.sh ${DATA_ROOT}/layout/${SHUTTLE_ID}.oas


# ※ Run cells before this cell for program installation and general setup.<br>Then choose one setup from the 38 designs below.<br>Finally, run the checks and examine the results

General results:
1. When the layout has cells that have the same name as primitive devices, netgen segfaults. <br>➡︎ Flatglob all layout cells with primitive device names.

# IV. Setup Chip Project

Checks will not pass with GDS or CDL file in the `TO_Apr2025` repo.
The following layout changes are necessary and will be automatically applied in this section.
1. Copy `TopMetal2/text 134/25` port text from cell `LNA` to top layout.
1. Copy corresponding `TopMetal2/pin 134/2` pin shapes from `LNA` to top layout (magic only).

Results:
*   Passes all checks with the above non-mask changes.

### Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env

if [[ ! -e ${MPW_ROOT}/layout/chip_list.txt ]]; then
  echo "Can't find chip_list.txt"
  exit 1
fi

mapfile -t chip_list < "${MPW_ROOT}/layout/chip_list.txt"

if [[ -z ${chip_list} ]]; then
  echo "$0: Undefined chip list"
  exit 1
fi

chip_list+=('Cancel')

PS3='Select number> '
trap "echo Exiting. Please input a number again; exit" SIGINT
select chip_name in "${chip_list[@]}"; do
  [[ -z "${REPLY}" ]] && {
    echo "Please enter a number."
    break
  }
  [[ -z "${chip_name}" ]] && {
    echo "##### ERROR: Invalid selection = ${REPLY}"
    break
  }
  if [[ "${chip_name}" == Cancel ]]; then
    echo "-> Job canceled"
    exit
  fi

  break 2
done

echo "export LAYOUT_TOP=${chip_name}"

{
  echo "export LAYOUT_TOP=${chip_name}"

  cat <<'EOF'
# export LAYOUT_TOP=2975_chip_top_0_6
export PROJECT=${LAYOUT_TOP}
export CHIP_ID=${LAYOUT_TOP%%_*}
export UPRJ_ROOT=${DATA_ROOT}/${CHIP_ID}
export WORK_ROOT=${DATA_ROOT}/${CHIP_ID}/work
export LAYOUT_DATA=${MPW_ROOT}/layout/${CHIP_ID}/${LAYOUT_TOP}.gds.gz
export SC_SPICE_FILES=${PDK_ROOT}/${PDK}/libs.ref/gf180mcu_fd_sc_mcu9t5v0/spice/*.spice
export SRAM_SPICE_FILES=${PDK_ROOT}/${PDK}/libs.ref/gf180mcu_ip_sram/spice/*.spice
export CDL_FILE=
export CARAVEL_ROOT=
export SRAM_MACRO=gf180mcu_fd_ip_sram__sram512x8m8wm1
EOF
} > "${DATA_ROOT}/project_env"

cat ${DATA_ROOT}/project_env
source ${DATA_ROOT}/project_env

if [[ ! -e ${LAYOUT_DATA} ]]; then
  echo "Can't find ${LAYOUT_DATA}"
  exit 1
fi

rm -f ${DATA_ROOT}/work
rm -rf ${WORK_ROOT}
mkdir -p ${WORK_ROOT}
ln -s ${WORK_ROOT} ${DATA_ROOT}/work
cd ${DATA_ROOT}/work
ln -s ${LAYOUT_DATA} .
ls -l


**Create `lvs_config.json` and update the `LVS_SPICE_FILES` and `LVS_VERILOG_FILES` for every design.**

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env

# Remove STD_CELL fill cells from the abstract
sed -i.bak '/__fill_/d; /__endcap/d; /__filltie/ s/^.*$/\t\t""/' ${LVS_ROOT}/tech/${PDK}/lvs_config.base.json

sed -e 's/\$SRAM_MACRO/gf180mcu_fd_ip_sram__sram64x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json > ${WORK_ROOT}/lvs_config.sram64.json
sed -e 's/\$SRAM_MACRO/gf180mcu_fd_ip_sram__sram128x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json > ${WORK_ROOT}/lvs_config.sram128.json
sed -e 's/\$SRAM_MACRO/gf180mcu_fd_ip_sram__sram256x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json > ${WORK_ROOT}/lvs_config.sram256.json
sed -e 's/\$SRAM_MACRO/gf180mcu_fd_ip_sram__sram512x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json > ${WORK_ROOT}/lvs_config.sram512.json

cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "STD_CELL_LIBRARY": "gf180mcu_fd_sc_mcu7t5v0",
  "INCLUDE_CONFIGS": [
     "$WORK_ROOT/lvs_config.sram64.json",
     "$WORK_ROOT/lvs_config.sram128.json",
     "$WORK_ROOT/lvs_config.sram256.json",
     "$WORK_ROOT/lvs_config.sram512.json",
     "$LVS_ROOT/tech/$PDK/lvs_config.chip_io.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "VIA_*",
    "GR_NMOS*",
    "*_route*",
    "*_via*",
    "via*",
    "TILE_*",
    "nfet*",
    "pfet*",
    "sealring*"
  ],
  "EXTRACT_ABSTRACT": [
    "gf180mcu_fd_io__cor*",
    "gf180mcu_fd_ip_sram*"
  ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SC_SPICE_FILES",
    "$SRAM_SPICE_FILES"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

**Create the cvcrc setup file and power file.**

In [ ]:
%%shell
cat /content/env
source /content/env
cat > ${WORK_ROOT}/cvcrc <<EOF
CVC_TOP = ${LAYOUT_TOP}
CVC_NETLIST = ${WORK_ROOT}/ext/${LAYOUT_TOP}.cdl.gz
CVC_MODEL_FILE = ${LVS_ROOT}/tech/${PDK}/cvc.models
CVC_POWER_FILE = ${WORK_ROOT}/cvc.power.${LAYOUT_TOP}
CVC_REPORT_FILE = ${WORK_ROOT}/cvc.log
EOF

cat > ${WORK_ROOT}/cvc.power.${PROJECT} <<EOF
VDD power 5.0
VSS power 0.0
*(gf180mcu_fd_ip_sram*)/CLK expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/CEN expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/A[*] expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/GWEN expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/WEN[*] expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/D[*] expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/Q[*] input min@VSS max@VDD
*(gf180mcu_fd_ip_sram*)/VDD expectSim@5.0
*(gf180mcu_fd_ip_sram*)/VSS expectSim@0.0
*_PAD* input min@VSS max@VDD
# bidir_* input min@VSS max@VDD
# input_* input min@VSS max@VDD
# clk_* input min@VSS max@VDD
# rst_* input min@VSS max@VDD
# analog_* input min@VSS max@VDD
EOF

# 2. 40_GHZ_LOW_NOISE_TIA

Checks will not pass with GDS or CDL file in the `TO_Apr2025` repo.
The following layout changes are necessary and will be automatically applied in this section.

1. Copy `TopMetal2/text 134/25` port text from cell `TIA` to top layout.
1. Copy corresponding `TopMetal2/pin 134/2` pin shapes from `TIA` to top layout (magic only).
1. The following text were moved to overlap pin shapes.
   ```
   VCC3 @ 506.235, 943.415
   GND @ 807.115, 943.415
   GND @ 693.765, 256.585
   GND @ 392.885, 256.585
   ```
The spice netlist used for LVS in this section was created from an xschem schematic available in the `TO_Apr2025/40_GHZ_LOW_NOISE_TIA/xschem` directory of the `ihp-mwp-be` repo. The schematic was created based on the `design_data/klayout/lvs/Schematic_of_netlist.png` diagram.

Results for 40_GHZ_LOW_NOISE_TIA
* `VCC1` and `VCC3` connections are reversed.![40GHz_klayout_lvs.png](https://github.com/d-m-bailey/ws-mpw-be/blob/main/TO_Apr2025/40_GHZ_LOW_NOISE_TIA/images/40GHz_klayout_lvs.png?raw=1)
* Reported 1 CVC error: This is the leak path from `RFIN` to `GND` through 2 resistors. Assumed to be intended.
```
    ! Short detected: 1257 to 0 Estimated current: 2.42mA
    /RX4 R rppd l=6u w=3u (r=520)
```

Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=40_GHZ_LOW_NOISE_TIA
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$PROJECT
export LAYOUT_TOP=FDM_QNC_00_LN_TIA
export ORIGINAL_DATA=$DATA_ROOT/$PROJECT/design_data/gds/FMD_QNC_00_40_GHz_Low_Noise_TIA.gds
export LAYOUT_DATA=$WORK_ROOT/$LAYOUT_TOP.gds
export SPICE_FILE=$CHECK_ROOT/$MPW/$PROJECT/xschem/lvs/$PROJECT.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create `lvs_config.json` and update the `LVS_SPICE_FILES` and `LVS_VERILOG_FILES` for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "cmim*",
    "npn13G2*",
    "rppd*",
    "rsil*",
    "sealring*",
    "*_STAGE",
    "DC_PAD_DOWN",
    "DC_PAD_UP",
    "GND",
    "GND_PLANE",
    "INPUT_PAD",
    "OUTPUT_PAD",
    "TIA",
    "VIA_STACK"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
GND power 0.0
VCC* power 3.3
RFIN input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Modify the layout file to pass LVS. (see explanation above.)

In [ ]:
%%shell
source /content/env
$CHECK_ROOT/scripts/apply_json_to_layout.py \
  --input_file=$ORIGINAL_DATA \
  --json_file=$CHECK_ROOT/$MPW/$PROJECT/gds/$PROJECT.changes.json \
  --output_file=$LAYOUT_DATA

---
# ※ Run BE checks


In [ ]:
%%shell
cat /content/env
source /content/env
# run_be_checks includes magic/netgen soft connection and LVS, and CVC-RV
# ${LVS_ROOT}/run_be_checks ${WORK_ROOT}/lvs_config.json
CVC_NETLIST=${WORK_ROOT}/ext/${LAYOUT_TOP}.cdl.gz

if [[ -e ${CVC_NETLIST} ]]; then
  ${LVS_ROOT}/run_cvc --noextract ${WORK_ROOT}/lvs_config.json
else
  ${LVS_ROOT}/run_cvc ${WORK_ROOT}/lvs_config.json
fi
echo "Exit code = $?"
exit 0

python3 $PDK_ROOT/$PDK/libs.tech/klayout/tech/lvs/run_lvs.py --layout=$LAYOUT_DATA \
  --netlist=$CDL_FILE \
  --run_dir=$WORK_ROOT/klayout \
  --topcell=$LAYOUT_TOP \
  --run_mode=deep \
  --spice_comments \
  --no_simplify \
  --combine_devices \
  --top_lvl_pins

LVS_BASENAME=$( echo $LAYOUT_DATA | sed -e 's,.*/,,' -e 's,\..*,,' )
$LVS_ROOT/check_klayout_ports $WORK_ROOT/klayout/$LVS_BASENAME.lvsdb > $WORK_ROOT/klayout/port_check.log
if [[ -e $WORK_ROOT/klayout/port_check.log ]]; then
  if [[ -s $WORK_ROOT/klayout/port_check.log ]]; then
    echo "* Klayout ports do not match *"
    echo "layout     source"
    awk '{printf "%-10s %s\n", $1, $2}' $WORK_ROOT/klayout/port_check.log
  else
    echo "Klayout ports match"
  fi
fi

# View verification logs

In [ ]:
from google.colab import files

files.view('/content/data/work/soft.log')
files.view('/content/data/work/lvs.log')
files.view('/content/data/work/cvc.log')

# View soft verification report

In [ ]:
from google.colab import files

files.view('/content/data/work/soft.report')

# View LVS report

In [ ]:
from google.colab import files

files.view('/content/data/work/lvs.report')

# View CVC report

In [ ]:
from google.colab import files

files.view('/content/data/work/cvc.error')